# Dana's Bakery Order Pipeline with LangGraph

This Colab notebook processes bakery orders with a deterministic LangGraph workflow. It uses ordinary Python rules only: no LLM calls, API keys, or model access. Each order returns status, final sold item, fulfillment type, total, a customer message, and an ordered list of graph steps.

The bakery asks before substituting. If the requested item is unavailable, an invocation without a customer answer returns a warm replacement offer. Pass customer replies through the optional replacement_answers input to accept or decline offers. A rejected/unconfirmed order is not booked and has no final sold item.


## 1. Install LangGraph if needed

LangGraph is the workflow library used to create and run the graph. The order decisions themselves use only local Python logic.


In [ ]:
# Install LangGraph only when it is not already available in this Colab runtime.
try:
    import langgraph  # noqa: F401
    print('LangGraph is already installed.')
except ImportError:
    !pip -q install langgraph


## 2. Define the order state and bakery rules

Today's inventory is shared by accepted orders during this runtime. Restarting the runtime resets it to the starting amounts.


In [ ]:
from typing import Literal, TypedDict

from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph


class OrderState(TypedDict, total=False):
    # Required customer-facing result fields:
    status: Literal['accepted', 'rejected']
    item: str | None
    delivery: Literal['pickup', 'delivery']
    total: float
    message: str
    steps: list[str]

    # Input fields:
    quantity: int | str
    day: str
    city: str
    replacement_answers: list[str | bool]

    # Fields used internally while the graph runs:
    valid: bool
    requested_item: str
    selected_item: str
    decision: str
    reason: str
    offer_item: str
    offer_number: int
    substitution: bool
    delivery_fee: int
    outside_delivery_area: bool
    goods_subtotal: float
    discount: float


MENU = {
    'challah': 18,
    'babka': 45,
    'box of rugelach': 30,
    'sourdough loaf': 25,
}
STARTING_INVENTORY = {
    'challah': 10,
    'babka': 0,
    'box of rugelach': 3,
    'sourdough loaf': 4,
}
INVENTORY = STARTING_INVENTORY.copy()

# Follow this replacement sequence, with no more than two offers to the customer.
REPLACEMENT = {
    'babka': 'box of rugelach',
    'box of rugelach': 'sourdough loaf',
    'sourdough loaf': 'challah',
}
WEEKDAYS = {'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'}
ALIASES = {
    'rugelach': 'box of rugelach',
    'box of rugelach': 'box of rugelach',
    'sourdough': 'sourdough loaf',
    'sourdough loaf': 'sourdough loaf',
}


## 3. Create the deterministic graph nodes

The nodes validate the order, check stock, handle customer-approved substitutions, decide delivery, calculate the total, update inventory, and write a reply. Every node appends its name to steps, so the result records the path in order.


In [ ]:
def add_step(state, step_name):
    """Return a new ordered list with this node recorded."""
    return [*state.get('steps', []), step_name]


def quantity_label(item: str, quantity: int) -> str:
    """Format product names naturally for customer-facing messages."""
    if item == 'box of rugelach':
        box_word = 'box' if quantity == 1 else 'boxes'
        return f'{quantity} {box_word} of rugelach'
    if item == 'sourdough loaf':
        loaf_word = 'loaf' if quantity == 1 else 'loaves'
        return f'{quantity} sourdough {loaf_word}'
    product_word = item if quantity == 1 else f'{item}s'
    return f'{quantity} {product_word}'


def validate_order(state: OrderState) -> dict:
    steps = add_step(state, 'validate_order')
    raw_item = str(state.get('item', '')).strip().casefold()
    item_names = {name.casefold(): name for name in MENU}
    requested_item = ALIASES.get(raw_item, item_names.get(raw_item))

    if raw_item in {'croissant', 'croissants'}:
        return {
            'steps': steps,
            'valid': False,
            'decision': 'invalid',
            'reason': 'Croissants are no longer made and cannot be ordered.',
            'requested_item': raw_item,
        }
    if requested_item is None:
        return {
            'steps': steps,
            'valid': False,
            'decision': 'invalid',
            'reason': 'That product is not on the current menu.',
            'requested_item': raw_item,
        }

    quantity = state.get('quantity')
    try:
        parsed_quantity = int(quantity)
        if isinstance(quantity, bool) or str(quantity).strip() != str(parsed_quantity) or parsed_quantity <= 0:
            raise ValueError
    except (TypeError, ValueError):
        return {
            'steps': steps,
            'valid': False,
            'decision': 'invalid',
            'reason': 'The quantity must be a positive whole number.',
            'requested_item': requested_item,
        }

    day = str(state.get('day', '')).strip().casefold()
    if day not in WEEKDAYS:
        return {
            'steps': steps,
            'valid': False,
            'decision': 'invalid',
            'reason': 'Please provide a valid day of the week.',
            'requested_item': requested_item,
        }

    return {
        'steps': steps,
        'valid': True,
        'requested_item': requested_item,
        'selected_item': requested_item,
        'quantity': parsed_quantity,
        'day': day.title(),
        'decision': 'ready_for_stock_check',
    }


def check_inventory(state: OrderState) -> dict:
    steps = add_step(state, 'check_inventory')
    if not state.get('valid'):
        return {'steps': steps}

    requested_item = state['requested_item']
    quantity = state['quantity']
    available = INVENTORY[requested_item]
    if available >= quantity:
        return {
            'steps': steps,
            'selected_item': requested_item,
            'decision': 'ready',
            'substitution': False,
        }
    return {
        'steps': steps,
        'decision': 'needs_substitution',
        'substitution': False,
    }


def handle_substitution(state: OrderState) -> dict:
    steps = add_step(state, 'handle_substitution')
    if state.get('decision') != 'needs_substitution':
        return {'steps': steps}

    requested_item = state['requested_item']
    quantity = state['quantity']
    answers = state.get('replacement_answers', [])
    current_item = requested_item
    offers_made = 0

    while offers_made < 2:
        alternative = REPLACEMENT.get(current_item)
        if alternative is None:
            break
        current_item = alternative

        # An item is offered only if stock can cover the entire order quantity.
        if INVENTORY[alternative] < quantity:
            continue

        offer_index = offers_made
        offers_made += 1
        answer = answers[offer_index] if offer_index < len(answers) else None
        if isinstance(answer, bool):
            answer = 'yes' if answer else 'no'
        if answer is not None:
            answer = str(answer).strip().casefold()

        if answer in {'yes', 'y'}:
            return {
                'steps': steps,
                'selected_item': alternative,
                'decision': 'ready',
                'substitution': True,
                'offer_item': alternative,
                'offer_number': offers_made,
            }
        if answer in {'no', 'n'}:
            continue

        # No answer yet: pause this order by returning a replacement offer.
        return {
            'steps': steps,
            'decision': 'awaiting_replacement_approval',
            'offer_item': alternative,
            'offer_number': offers_made,
            'substitution': False,
        }

    return {
        'steps': steps,
        'decision': 'cancelled',
        'substitution': False,
    }


def choose_delivery(state: OrderState) -> dict:
    steps = add_step(state, 'choose_delivery')
    city = str(state.get('city', '')).strip()
    if city.casefold() == 'jerusalem':
        return {
            'steps': steps,
            'delivery': 'delivery',
            'delivery_fee': 20,
            'outside_delivery_area': False,
        }
    return {
        'steps': steps,
        'delivery': 'pickup',
        'delivery_fee': 0,
        'outside_delivery_area': bool(city),
    }


def calculate_total(state: OrderState) -> dict:
    steps = add_step(state, 'calculate_total')
    if state.get('decision') != 'ready':
        return {'steps': steps, 'goods_subtotal': 0, 'discount': 0, 'total': 0.0}

    goods_subtotal = MENU[state['selected_item']] * state['quantity']
    discount = 0
    if goods_subtotal > 150 and str(state['day']).casefold() != 'friday':
        discount = round(goods_subtotal * 0.15, 2)
    total = round(goods_subtotal - discount + state.get('delivery_fee', 0), 2)
    return {
        'steps': steps,
        'goods_subtotal': goods_subtotal,
        'discount': discount,
        'total': total,
    }


def update_inventory(state: OrderState) -> dict:
    steps = add_step(state, 'update_inventory')
    if state.get('decision') == 'ready':
        sold_item = state['selected_item']
        INVENTORY[sold_item] -= state['quantity']
        return {'steps': steps, 'status': 'accepted', 'item': sold_item}
    return {'steps': steps, 'status': 'rejected', 'item': None}


def write_customer_reply(state: OrderState) -> dict:
    steps = add_step(state, 'write_customer_reply')
    requested_item = state.get('requested_item', state.get('item', 'that item'))
    city = str(state.get('city', '')).strip()

    if state.get('decision') == 'invalid':
        message = f"Hi there! I couldn't understand this order: {state.get('reason', 'please check the details')}. Please send it again and I'll be happy to help. —Dana"
    elif state.get('decision') == 'awaiting_replacement_approval':
        alternative = state['offer_item']
        delivery_note = ' If you accept, it will be delivered in Jerusalem for ₪20.' if state.get('delivery') == 'delivery' else (' We do not deliver outside Jerusalem, so an accepted order would be free shop pickup.' if city else ' An accepted order would be free shop pickup.')
        message = (f"Hi there! We're out of {requested_item}. Would you like {quantity_label(alternative, state['quantity'])} instead "
                   f"at ₪{MENU[alternative]} each?{delivery_note} Please reply yes or no; your order is not placed until you accept. —Dana")
    elif state.get('decision') == 'cancelled':
        message = f"I'm so sorry, but we couldn't fill your {quantity_label(requested_item, state.get('quantity', 1))} order with an available replacement after up to two offers. I've cancelled it. —Dana"
    else:
        selected_item = state['selected_item']
        city_note = ''
        if state.get('outside_delivery_area'):
            city_note = f' We do not deliver to {city}, so I switched your order to free shop pickup.'
        if state.get('substitution'):
            substitution_note = f" We were out of {requested_item}; with your approval, I substituted {quantity_label(selected_item, state['quantity'])} and kept your original quantity."
        else:
            substitution_note = ''
        discount_note = f" Your 15% baked-goods discount is ₪{state['discount']:.2f}." if state.get('discount', 0) else ''
        delivery_note = ' Jerusalem delivery is ₪20.' if state.get('delivery') == 'delivery' else ''
        message = (f"Hi there! Thanks for ordering {quantity_label(selected_item, state['quantity'])} from Dana's Bakery."
                   f"{substitution_note}{city_note}{discount_note}{delivery_note} Your total is ₪{state['total']:.2f}. See you soon! —Dana")

    return {'steps': steps, 'message': message}


## 4. Build and visualize the graph

The PNG visualization is rendered directly in the notebook as required.


In [ ]:
builder = StateGraph(OrderState)
builder.add_node('validate_order', validate_order)
builder.add_node('check_inventory', check_inventory)
builder.add_node('handle_substitution', handle_substitution)
builder.add_node('choose_delivery', choose_delivery)
builder.add_node('calculate_total', calculate_total)
builder.add_node('update_inventory', update_inventory)
builder.add_node('write_customer_reply', write_customer_reply)

builder.add_edge(START, 'validate_order')
builder.add_edge('validate_order', 'check_inventory')
builder.add_edge('check_inventory', 'handle_substitution')
builder.add_edge('handle_substitution', 'choose_delivery')
builder.add_edge('choose_delivery', 'calculate_total')
builder.add_edge('calculate_total', 'update_inventory')
builder.add_edge('update_inventory', 'write_customer_reply')
builder.add_edge('write_customer_reply', END)

graph = builder.compile()


In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## 5. Run the requested example

This exact invocation has no replacement answer, so the graph returns a rejected, unbooked order and asks whether the customer wants the available rugelach. It still returns all six required keys. An unaccepted or cancelled order returns item=None and total=0 because no item was sold.


In [ ]:
sample = graph.invoke({"item": "babka", "quantity": 2, "day": "Thursday", "city": "Jerusalem"})
required_keys = ['status', 'item', 'delivery', 'total', 'message', 'steps']
{key: sample[key] for key in required_keys}


## 6. Record the customer's answer

To accept the offered rugelach, invoke the same order with replacement_answers=['yes']. Use ['no'] to decline the first offer; the graph will ask about the next in-stock item in the sequence. Use ['no', 'no'] to decline two offers and cancel. Answers are processed deterministically, with no interactive input or model calls inside the graph.


In [ ]:
approved_order = graph.invoke({
    'item': 'babka',
    'quantity': 2,
    'day': 'Thursday',
    'city': 'Jerusalem',
    'replacement_answers': ['yes'],
})
{key: approved_order[key] for key in required_keys}
